<a href="https://colab.research.google.com/github/rahulpirwani7/24k-0657/blob/main/HeartPredicationWithTrees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
import os
import pandas as pd
import kagglehub
import numpy as np

def load_dataset(path):
    """
    Recursively searches for the first CSV file in the given directory
    and returns it as a pandas DataFrame.
    """

    for root, dirs, files in os.walk(path):
        for file in files:
            if file.endswith(".csv"):
                file_path = os.path.join(root, file)
                df=pd.read_csv(file_path)

                if df.shape[1] == 1:
                    df = pd.read_csv(file_path, sep=";")
                return df

    raise FileNotFoundError("No CSV file found in the dataset.")

In [40]:
path1 = kagglehub.dataset_download("alexteboul/heart-disease-health-indicators-dataset")
path2 = kagglehub.dataset_download("kamilpytlak/personal-key-indicators-of-heart-disease")
path3 = kagglehub.dataset_download("sulianova/cardiovascular-disease-dataset")

df1,df2,df3=load_dataset(path1),load_dataset(path2),load_dataset(path3)

Using Colab cache for faster access to the 'heart-disease-health-indicators-dataset' dataset.
Using Colab cache for faster access to the 'personal-key-indicators-of-heart-disease' dataset.
Using Colab cache for faster access to the 'cardiovascular-disease-dataset' dataset.


In [41]:
#  features for my  model : Age,Sex,Smoking,Stroke,. Alcohol_Consumption and Physical_Activity

df1.rename(columns={
    "Smoker":"Smoking",
    'HvyAlcoholConsump':'Alcohol_Consumption',
    'PhysActivity':'Physical_Activity',
    'HeartDiseaseorAttack':'HeartDisease'
},inplace=True)

df2.rename(columns={
    'AlcoholDrinking':'Alcohol_Consumption',
    'PhysicalHealth':'Physical_Activity',
    'AgeCategory':'Age',
},inplace=True)

df3.rename(columns={
    'age':'Age',
    'gender':'Sex',
    'smoke':'Smoking',
    'alco':'Alcohol_Consumption',
    'active':'Physical_Activity',
    'cardio':'HeartDisease'
},inplace=True)



In [42]:
df1=df1[['Age','Sex','Smoking','Stroke','Alcohol_Consumption','Physical_Activity','HeartDisease']]
df2=df2[['Age','Sex','Smoking','Stroke','Alcohol_Consumption','Physical_Activity','HeartDisease']]
df3=df3[['Age','Sex','Smoking','Alcohol_Consumption','Physical_Activity','HeartDisease']]

In [43]:

df1.loc[df1['Age'] != 1, 'Age'] = df1.loc[df1['Age'] != 1, 'Age'] * 5 + 17
df1.loc[df1['Age'] == 1, 'Age'] = 21

age_map = {
    "18-24": 21,
    "25-29": 27,
    "30-34": 32,
    "35-39": 37,
    "40-44": 42,
    "45-49": 47,
    "50-54": 52,
    "55-59": 57,
    "60-64": 62,
    "65-69": 67,
    "70-74": 72,
    "75-79": 77,
    "80 or older": 85
}

df2["Age"] = df2["Age"].map(age_map)




In [44]:
df2.loc[df2['Sex'] == 'Female', 'Sex'] = 0
df2.loc[df2['Sex'] == 'Male', 'Sex'] = 1
df2.loc[df2['Smoking'] == 'Yes', 'Smoking'] = 1
df2.loc[df2['Smoking'] == 'No', 'Smoking'] = 0
df2.loc[df2['Stroke'] == 'Yes', 'Stroke'] = 1
df2.loc[df2['Stroke'] == 'No', 'Stroke'] = 0
df2.loc[df2['Alcohol_Consumption'] == 'Yes', 'Alcohol_Consumption'] = 1
df2.loc[df2['Alcohol_Consumption'] == 'No', 'Alcohol_Consumption'] = 0
df2.loc[df2['HeartDisease'] == 'Yes', 'HeartDisease'] = 1
df2.loc[df2['HeartDisease'] == 'No', 'HeartDisease'] = 0
df2["Physical_Activity"] = (df2["Physical_Activity"] > 10).astype(int)

In [45]:
df3['Age']=df3['Age']/365
df3['Sex']=df3['Sex']-1
df3["Stroke"] = np.random.randint(0, 2, size=len(df3))

In [46]:
import pandas as pd

df = pd.concat([df1,df2, df3], axis=0, ignore_index=True)

In [47]:
df=pd.concat([
    df[df["HeartDisease"] == 1],
    df[df["HeartDisease"] == 0].sample(n=110000, random_state=42)
])

In [48]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(df.drop("HeartDisease", axis=1), df["HeartDisease"], test_size=0.4, random_state=42)
X_cross, X_test, y_cross, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [49]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model = DecisionTreeClassifier(random_state=42)


In [50]:
y_cross=y_cross.astype(int)
y_test=y_test.astype(int)
y_train=y_train.astype(int)

In [51]:
model.fit(X_train, y_train)
y_pred = model.predict(X_cross)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.56      0.64      0.60     22026
           1       0.44      0.36      0.39     17223

    accuracy                           0.52     39249
   macro avg       0.50      0.50      0.50     39249
weighted avg       0.51      0.52      0.51     39249

